# Step 1: Multi-Scholar Dataset Cleaning & Preprocessing

This notebook cleans and standardizes raw Islamic fatwa datasets from trusted Sunni/Salafi scholars and official bodies:
1. **Sheikh Muhammad ibn Saalih al-Uthaymeen** (الشيخ محمد بن صالح العثيمين)
2. **Sheikh Saleh Al-Fawzan** (الشيخ صالح الفوزان)
3. **Al-Lajnah Ad-Da'imah lil-Ifta / Council of Senior Scholars** (اللجنة الدائمة للإفتاء / هيئة كبار العلماء)
4. **Sheikh Abdulaziz Aal ash-Sheikh** (Grand Mufti of Saudi Arabia)
5. **Sheikh Abdullah al-Manee'** (Council of Senior Scholars)
6. **Sheikh Saleh bin Humaid** (Council of Senior Scholars)
7. Merging with existing cleaned fatwas of **Sheikh Abdulaziz ibn Baz** (الشيخ عبدالعزيز بن باز)

### Key Improvements & Fixes over Raw Scrapings:
- **Title Extraction for Ibn Uthaymeen**: Resolves a major scraping flaw where ~90% of raw titles were generic category paths (e.g. `الطهارة >> المياه`), while the true specific fatwa title was stored inside the category metadata.
- **Breadcrumb Hierarchy Cleanup for Al-Lajnah Ad-Da'imah**: Strips administrative portal prefixes (`تصفح برقم المجلد >> ...`) to extract clear, focused fatwa titles while retaining the volume/subject hierarchy.
- **Empty / Incomplete Record Filtering**: Drops corrupted entries lacking substantial answer text.
- **Text Normalization**: Unicode NFKC normalization, `ftfy` mojibake repair, regex stripping of HTML entities and footnote markers without stripping essential Arabic diacritics or words.
- **Unified Schema**: Produces standardized CSVs matching `['fatwa_id', 'source', 'title', 'question', 'answer', 'links', 'categories', 'embed_text']` and combines them into `multi_scholar_clean.csv`.

In [1]:
# Setup & Imports
import os
import re
import json
import unicodedata
import ast
from pathlib import Path
import pandas as pd
import ftfy

print('Pandas version:', pd.__version__)

Pandas version: 3.0.5


In [2]:
# Normalization & Cleaning Functions

def clean_text(s: str) -> str:
    """Robust light-touch cleaning for Arabic text:
    - ftfy text fix
    - NFKC Unicode normalization
    - Strip HTML entities and scraping artifacts
    - Strip footnote markers [1], [2]
    - Strip zero-width characters and BOM
    - Normalize whitespace while preserving Arabic script integrity
    """
    if not isinstance(s, str):
        return ''
    s = ftfy.fix_text(s)
    s = unicodedata.normalize('NFKC', s)
    s = re.sub(r'&[a-zA-Z]+;|&#\d+;', ' ', s)
    s = re.sub(r'\[\d+\]', ' ', s)
    s = re.sub(r'[\u200b-\u200f\ufeff]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

In [3]:
# Dataset Processor

def process_raw_dataset(source_label: str, raw_json_path: str, title_strategy: str = 'direct') -> pd.DataFrame:
    with open(raw_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    rows = []
    for item in data:
        ans = clean_text(item.get('answer', ''))
        if not ans or len(ans) < 5:
            continue
        
        raw_title = clean_text(item.get('title', ''))
        raw_cats = item.get('categories', [])
        if isinstance(raw_cats, str):
            try:
                raw_cats = ast.literal_eval(raw_cats)
            except Exception:
                raw_cats = [raw_cats] if raw_cats.strip() else []
        if not isinstance(raw_cats, list):
            raw_cats = []
        
        if title_strategy == 'uthaymeen':
            # Extract specific fatwa title from categories if present
            specific = [clean_text(c) for c in raw_cats if c != 'article' and '>>' not in c and len(c.strip()) > 3]
            if specific:
                final_title = specific[0]
                cat_list = [clean_text(c) for c in raw_cats if c != 'article' and c != final_title]
            else:
                parts = [p.strip() for p in raw_title.split('>>') if p.strip()]
                final_title = parts[-1] if parts else raw_title
                cat_list = parts[:-1]
        elif title_strategy == 'lajnah':
            parts = [p.strip() for p in raw_title.split('>>') if p.strip()]
            if parts and parts[0].startswith('تصفح'):
                parts = parts[1:]
            if len(parts) >= 2:
                final_title = parts[-1]
                cat_list = parts[:-1]
            elif len(parts) == 1:
                final_title = parts[0]
                cat_list = []
            else:
                final_title = raw_title
                cat_list = []
        else:
            final_title = raw_title
            cat_list = [clean_text(c) for c in raw_cats if clean_text(c)]
        
        q = clean_text(item.get('question', ''))
        link = clean_text(item.get('link', ''))
        embed_text = f"{final_title} — {ans}"
        
        rows.append({
            'source': source_label,
            'title': final_title,
            'question': q,
            'answer': ans,
            'links': link,
            'categories': str(cat_list),
            'embed_text': embed_text
        })
    
    df = pd.DataFrame(rows).drop_duplicates(subset=['title', 'answer']).reset_index(drop=True)
    df.insert(0, 'fatwa_id', df.index)
    return df

In [4]:
# Clean All Scholar Datasets
DATA_RAW = Path('../data/raw') if Path('../data/raw').exists() else Path('data/raw')
DATA_OUT = Path('../data/cleaned') if Path('../data/cleaned').exists() else Path('data/cleaned')
DATA_OUT.mkdir(parents=True, exist_ok=True)

scholars_config = [
    ('Ibn Uthaymeen', DATA_RAW / 'fatawa_uthaymeen.json', 'uthaymeen', 'uthaymeen_clean.csv'),
    ('Saleh Al-Fawzan', DATA_RAW / 'fatawa_fawzan.json', 'direct', 'fawzan_clean.csv'),
    ('Al-Lajnah Ad-Daimah', DATA_RAW / 'fatawa_lajnah_daimah.json', 'lajnah', 'lajnah_clean.csv'),
    ('Abdulaziz Aal ash-Sheikh', DATA_RAW / 'fatawa_mufti_al_sheikh.json', 'direct', 'mufti_clean.csv'),
    ('Abdullah al-Manee', DATA_RAW / 'fatawa_al_manee.json', 'direct', 'al_manee_clean.csv'),
    ('Saleh bin Humaid', DATA_RAW / 'fatawa_bin_humaid.json', 'direct', 'bin_humaid_clean.csv'),
]

cleaned_dfs = []
for label, path, strat, out_name in scholars_config:
    if path.exists():
        df_scholar = process_raw_dataset(label, str(path), title_strategy=strat)
        df_scholar.to_csv(DATA_OUT / out_name, index=False)
        cleaned_dfs.append(df_scholar)
        print(f"{label:25s}: {len(df_scholar):,} rows -> {out_name}")
    else:
        print(f"! Missing {path}")

Ibn Uthaymeen            : 9,119 rows -> uthaymeen_clean.csv


Saleh Al-Fawzan          : 1,714 rows -> fawzan_clean.csv


Al-Lajnah Ad-Daimah      : 9,086 rows -> lajnah_clean.csv
Abdulaziz Aal ash-Sheikh : 135 rows -> mufti_clean.csv


Abdullah al-Manee        : 233 rows -> al_manee_clean.csv
Saleh bin Humaid         : 48 rows -> bin_humaid_clean.csv


In [5]:
# Combine with Existing Bin Baz Corpus
binbaz_path = Path('../data/binbaz_clean.csv') if Path('../data/binbaz_clean.csv').exists() else Path('data/binbaz_clean.csv')
if binbaz_path.exists():
    df_binbaz = pd.read_csv(binbaz_path)
    print(f"Ibn Baz (existing):       {len(df_binbaz):,} rows")
    all_scholars_df = [df_binbaz] + cleaned_dfs
else:
    all_scholars_df = cleaned_dfs

master_df = pd.concat([
    d[['source', 'title', 'question', 'answer', 'links', 'categories', 'embed_text']]
    for d in all_scholars_df
], ignore_index=True)

master_df.insert(0, 'fatwa_id', master_df.index)
master_csv = DATA_OUT / 'multi_scholar_clean.csv'
master_df.to_csv(master_csv, index=False)

print("\nSuccessfully generated Master Multi-Scholar Corpus: " + f"{len(master_df):,} total fatwas!")
print("Saved to:", master_csv)
print("\nBreakdown by Scholar:")
print(master_df['source'].value_counts())

Ibn Baz (existing):       19,727 rows



Successfully generated Master Multi-Scholar Corpus: 40,062 total fatwas!
Saved to: ../data/cleaned/multi_scholar_clean.csv

Breakdown by Scholar:
source
Ibn Baz                     19727
Ibn Uthaymeen                9119
Al-Lajnah Ad-Daimah          9086
Saleh Al-Fawzan              1714
Abdullah al-Manee             233
Abdulaziz Aal ash-Sheikh      135
Saleh bin Humaid               48
Name: count, dtype: int64


In [6]:
# Corpus Verification
print("Missing values in master corpus:")
print(master_df[['source', 'title', 'answer', 'embed_text']].isnull().sum())
print("\nSample preview:")
display_cols = ['fatwa_id', 'source', 'title']
print(master_df[display_cols].sample(5, random_state=42).to_string(index=False))

Missing values in master corpus:
source         0
title          0
answer        75
embed_text     0
dtype: int64

Sample preview:
 fatwa_id        source                                            title
    28516 Ibn Uthaymeen تاب من غيبة الآخرين وتصدق عليهم فهل تقبل توبته ؟
     6648       Ibn Baz                     مشروعية عقد النكاح في المسجد
    11770       Ibn Baz                      حكم شكر الإنسان على الإحسان
     9702       Ibn Baz                     حكم صيام الكتابي إذا لم يسلم
    27969 Ibn Uthaymeen                                     فتاوى المرأة
